# exp_009 — Phase 1: Curriculum sampling

**Purpose:** for each of ~926 training prompts, generate N samples with the **base** Qwen3-4B-Thinking-2507 and record how many are correct. Filter to the "sweet spot" (1 ≤ correct < N) where GRPO can produce gradient signal.

**Why:** the pilot showed ~80% of GRPO compute is wasted on uniform-reward batches (all wrong → too hard, all right → too easy). Curriculum filtering pre-identifies the 30-40% of prompts where the model gets some right and some wrong. Training only on those means every batch produces gradient.

**Runtime:** Colab Pro — A100 40GB. ~1.5 hrs with vLLM (4 samples × 926 prompts).

**Output:** `/content/difficulty_samples.jsonl` (full data) and `/content/sweet_spot_ids.json` (filtered IDs to feed into training).

In [ ]:
# Cell 1 — Install vLLM (standalone inference, no TRL/GRPO integration)
#
# Why this should be less brittle than past attempts: we're not pinning vllm against
# trl/peft/bitsandbytes — just plain inference. The past dependency hell was vllm + TRL
# fighting over torch versions; here vllm is alone.
#
# CRITICAL: After this cell, do Runtime → Restart session before continuing.
!pip install -q vllm
!pip install -q sympy antlr4-python3-runtime==4.11

print("\n✅ Installs done. Runtime → Restart session, then run from Cell 2.")

In [ ]:
# Cell 2 — Config
MODEL_NAME = "Qwen/Qwen3-4B-Thinking-2507"

# Sampling — keep aligned with training so difficulty is representative
N_SAMPLES_PER_PROMPT = 4   # 4 is enough signal: 0/4 hard, 1-3/4 sweet, 4/4 easy
MAX_PROMPT_LEN       = 1024
MAX_COMPLETION_LEN   = 4096
TEMPERATURE          = 1.0     # match GRPO training temp
TOP_P                = 0.95
TOP_K                = 20

# vLLM engine sizing for A100 40GB
VLLM_MAX_MODEL_LEN   = 5120    # prompt (~1024) + completion (4096) headroom
VLLM_MAX_NUM_SEQS    = 32      # batch size — A100 40GB easily handles this for 4B in fp16
VLLM_GPU_UTIL        = 0.90

# Pilot mode — verify pipeline on a tiny slice first
PILOT_MODE = True
PILOT_N    = 16

RANDOM_SEED = 42

OUTPUT_PATH    = "/content/difficulty_samples.jsonl"
FILTERED_PATH  = "/content/sweet_spot_ids.json"

print(f"PILOT_MODE: {PILOT_MODE}  ({'16 prompts' if PILOT_MODE else 'full ~926 prompts'})")
print(f"N_SAMPLES_PER_PROMPT: {N_SAMPLES_PER_PROMPT}")
print(f"Model: {MODEL_NAME}")

In [ ]:
# Cell 3 — Upload competition files
import os
from google.colab import files

REQUIRED = [
    ("/content/public.jsonl",  "data/public.jsonl"),
    ("/content/dev.jsonl",     "data/splits/dev.jsonl"),
    ("/content/judger.py",     "judger.py (repo root)"),
    ("/content/utils.py",      "utils.py (repo root)"),
]
for path, label in REQUIRED:
    if not os.path.exists(path):
        print(f"Upload: {label}")
        files.upload()
    else:
        print(f"Already present: {path}")

import sys
sys.path.insert(0, "/content")
from judger import Judger
JUDGER = Judger()
print("Judger loaded.")

In [ ]:
# Cell 4 — Prompts (mirror the training notebook exactly so difficulty estimates
# reflect the same context the model sees during GRPO training).
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

FEWSHOT_MATH = []

FEWSHOT_MCQ = [
    {"role": "user", "content": (
        "Find 1 over 6 + 1 over 8.\n\n"
        "Options:\nA. 7 over 24\nB. 2 over 14\nC. 1 over 4\nD. 7 over 48\n"
        "E. 2 over 24\nF. 1 over 14\nG. 1 over 2\nH. 8 over 14\nI. 0.21 Repeating\nJ. 4 over 24"
    )},
    {"role": "assistant", "content": "Common denominator is 24: 1/6 = 4/24 and 1/8 = 3/24, so 4/24 + 3/24 = 7/24. \\boxed{A}"},
    {"role": "user", "content": (
        "The function value of $\\cos(\\pi + 5i)$ is ( ).\n\n"
        "Options:\nA. -cosh5\nB. -sinh5\nC. sin5i\nD. -sin5\nE. cos5\n"
        "F. cosh5i\nG. sinh5\nH. -cos5\nI. cosh5\nJ. -sin5i"
    )},
    {"role": "assistant", "content": "$\\cos(\\pi + 5i) = -\\cos(5i) = -\\cosh(5)$, using $\\cos(\\pi+x) = -\\cos x$ and $\\cos(ix) = \\cosh x$. \\boxed{A}"},
    {"role": "user", "content": (
        "Find the range of $f(x) = \\frac{ x }{ 1+x^2 }$.\n\n"
        "Options:\nA. -\\frac{1}{3} \\le y \\le \\frac{1}{3}\nB. -\\frac{1}{\\sqrt{3}} \\le y \\le \\frac{1}{\\sqrt{3}}\n"
        "C. -\\frac{1}{4} \\le y \\le \\frac{1}{4}\nD. -\\frac{1}{\\sqrt{2}} \\le y \\le \\frac{1}{\\sqrt{2}}\n"
        "E. -\\frac{1}{\\sqrt{6}} \\le y \\le \\frac{1}{\\sqrt{6}}\nF. -\\frac{1}{2} \\le y \\le \\frac{1}{2}\n"
        "G. -\\frac{1}{\\sqrt{5}} \\le y \\le \\frac{1}{\\sqrt{5}}\nH. -1 \\le y \\le 1\n"
        "I. -\\frac{1}{\\sqrt{7}} \\le y \\le \\frac{1}{\\sqrt{7}}\nJ. -\\frac{1}{\\sqrt{4}} \\le y \\le \\frac{1}{\\sqrt{4}}"
    )},
    {"role": "assistant", "content": "$f'(x) = \\frac{1-x^2}{(1+x^2)^2} = 0$ at $x=\\pm 1$, giving $f(\\pm 1) = \\pm 1/2$. So range is $-1/2 \\le y \\le 1/2$. \\boxed{F}"},
]

In [ ]:
# Cell 5 — Build prompt list (public.jsonl minus dev split)
import json, random

random.seed(RANDOM_SEED)
LETTERS = "ABCDEFGHIJ"

dev_ids = set()
with open("/content/dev.jsonl") as f:
    for line in f:
        dev_ids.add(json.loads(line)["id"])
print(f"Dev IDs (excluded from sampling): {len(dev_ids)}")

prompts = []
with open("/content/public.jsonl") as f:
    for line in f:
        ex = json.loads(line)
        if ex["id"] in dev_ids:
            continue
        is_mcq = bool("options" in ex and ex["options"])
        question_text = ex["question"]
        if is_mcq:
            opts = "\n".join(f"{LETTERS[i]}. {v}" for i, v in enumerate(ex["options"]))
            question_text = f"{question_text}\n\nOptions:\n{opts}"
            sys_prompt = SYSTEM_PROMPT_MCQ
            fewshots = FEWSHOT_MCQ
        else:
            sys_prompt = SYSTEM_PROMPT_MATH
            fewshots = FEWSHOT_MATH

        msgs = [{"role": "system", "content": sys_prompt}]
        msgs.extend(fewshots)
        msgs.append({"role": "user", "content": question_text})

        prompts.append({
            "id": ex["id"],
            "msgs": msgs,
            "answer": ex["answer"],
            "options": ex.get("options", []),
            "is_mcq": is_mcq,
        })

random.shuffle(prompts)
if PILOT_MODE:
    prompts = prompts[:PILOT_N]
    print(f"PILOT_MODE — {len(prompts)} prompts")
else:
    print(f"Full mode — {len(prompts)} prompts")

print("MCQ:", sum(p["is_mcq"] for p in prompts), "  Free-form:", sum(not p["is_mcq"] for p in prompts))

In [ ]:
# Cell 6 — Initialize vLLM engine
#
# Colab+vLLM compatibility patches (proven in colab_dev.ipynb):
#   1. VLLM_USE_V1=0 → use V0 engine. The V1 EngineCore subprocess inherits
#      Colab's ipykernel iostream and crashes on suppress_stdout's fileno() call.
#   2. VLLM_ENABLE_V1_MULTIPROCESSING=0 → defense in depth, skip subprocess path.
#   3. _StdProxy: gives sys.stdout/stderr a real fileno() (so vLLM's
#      suppress_stdout works) while keeping write/flush routed through ipykernel
#      (so print() output still reaches Colab's UI).
# All three MUST happen BEFORE `import vllm`.
import os, sys, io as _io

os.environ["VLLM_USE_V1"]                    = "0"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["PYTORCH_ALLOC_CONF"]             = "expandable_segments:True"

class _StdProxy:
    """sys.stdout/stderr proxy: forwards write/flush to original (ipykernel)
    so prints show up in Colab UI, but reports a real OS fd via fileno()
    so vLLM's suppress_stdout doesn't crash."""
    def __init__(self, original, fd):
        self._original = original
        self._fd = fd
    def fileno(self):           return self._fd
    def write(self, s):         return self._original.write(s)
    def flush(self):            return self._original.flush()
    def isatty(self):           return False
    def writable(self):         return True
    @property
    def closed(self):           return False
    def __getattr__(self, name): return getattr(self._original, name)

for _name, _fd in [("stdout", 1), ("stderr", 2)]:
    s = getattr(sys, _name)
    try:
        s.fileno()
    except _io.UnsupportedOperation:
        setattr(sys, _name, _StdProxy(s, _fd))

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Render all chat templates to plain strings (vLLM takes strings).
rendered_prompts = []
for p in prompts:
    rendered = tokenizer.apply_chat_template(
        p["msgs"], tokenize=False, add_generation_prompt=True
    )
    rendered_prompts.append(rendered)

print(f"Rendered {len(rendered_prompts)} prompts.")
print("Last 200 chars of first prompt:", repr(rendered_prompts[0][-200:]))

print("\nLoading vLLM engine (this takes 2-3 min — compiles CUDA kernels)...")
llm = LLM(
    model=MODEL_NAME,
    dtype="float16",
    max_model_len=VLLM_MAX_MODEL_LEN,
    max_num_seqs=VLLM_MAX_NUM_SEQS,
    gpu_memory_utilization=VLLM_GPU_UTIL,
    trust_remote_code=True,
    enable_prefix_caching=False,
    seed=RANDOM_SEED,
)

sampling_params = SamplingParams(
    n=N_SAMPLES_PER_PROMPT,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_COMPLETION_LEN,
    seed=RANDOM_SEED,
)
print("vLLM ready.")

In [ ]:
# Cell 7 — Generate + score with resume support
# vLLM batches all prompts at once via PagedAttention. For 926 prompts × 4 samples
# we issue ONE call to llm.generate() and let vLLM schedule.
import time, json, os

# Resume support: skip prompt IDs already saved.
done_ids = set()
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH) as f:
        for line in f:
            done_ids.add(json.loads(line)["id"])
    print(f"Resuming — {len(done_ids)} prompts already sampled")

remaining_idx = [i for i, p in enumerate(prompts) if p["id"] not in done_ids]
remaining_prompts = [rendered_prompts[i] for i in remaining_idx]
remaining_meta    = [prompts[i] for i in remaining_idx]

if not remaining_prompts:
    print("Nothing to do — all prompts already sampled.")
else:
    print(f"Generating {N_SAMPLES_PER_PROMPT} samples × {len(remaining_prompts)} prompts...")
    t0 = time.time()
    outputs = llm.generate(remaining_prompts, sampling_params)
    gen_time = time.time() - t0
    print(f"\nGeneration done in {gen_time/60:.1f} min ({gen_time/len(remaining_prompts):.1f} sec/prompt).")

    # Score and save incrementally
    print("Scoring...")
    n_written = 0
    with open(OUTPUT_PATH, "a") as fout:
        for meta, out in zip(remaining_meta, outputs):
            n_correct = 0
            sample_records = []
            for sample in out.outputs:
                text = sample.text
                # Parse answer after </think>
                idx = text.rfind("</think>")
                post = text[idx + len("</think>"):] if idx >= 0 else text

                gold = meta["answer"]
                opts = meta["options"]
                opts_list = ([opts] * len(gold)) if gold else [None]
                try:
                    ok = JUDGER.auto_judge(post, gold, opts_list)
                except Exception:
                    ok = False
                if ok:
                    n_correct += 1
                sample_records.append({
                    "text": text,
                    "correct": ok,
                    "clipped": len(sample.token_ids) >= MAX_COMPLETION_LEN,
                })

            record = {
                "id": meta["id"],
                "is_mcq": meta["is_mcq"],
                "num_correct": n_correct,
                "n_samples": N_SAMPLES_PER_PROMPT,
                "completions": sample_records,
            }
            fout.write(json.dumps(record) + "\n")
            n_written += 1
            if n_written % 50 == 0:
                print(f"  scored {n_written}/{len(remaining_meta)}")
    print(f"\nDone. Wrote {n_written} new records to {OUTPUT_PATH}")
    print(f"Total wall time (gen + score): {(time.time()-t0)/60:.1f} min")

In [ ]:
# Cell 8 — Difficulty distribution + filtered ID export
import json
from collections import Counter

records = []
with open(OUTPUT_PATH) as f:
    for line in f:
        records.append(json.loads(line))
print(f"Total prompts sampled: {len(records)}")

dist = Counter(r["num_correct"] for r in records)
print(f"\nDifficulty distribution (correct / {N_SAMPLES_PER_PROMPT}):")
for k in range(N_SAMPLES_PER_PROMPT + 1):
    n = dist.get(k, 0)
    pct = 100 * n / len(records)
    bar = "\u2588" * int(pct / 2)
    print(f"  {k}/{N_SAMPLES_PER_PROMPT}: {n:4d} ({pct:5.1f}%) {bar}")

# Sweet spot: at least one correct AND at least one wrong → guaranteed gradient
sweet_spot = [r for r in records if 1 <= r["num_correct"] < N_SAMPLES_PER_PROMPT]
print(f"\nSweet-spot count (1 <= correct < {N_SAMPLES_PER_PROMPT}): {len(sweet_spot)} "
      f"({100*len(sweet_spot)/len(records):.1f}%)")

# Breakdown by question type
mcq = [r for r in records if r["is_mcq"]]
ff  = [r for r in records if not r["is_mcq"]]
if mcq:
    print(f"\nMCQ ({len(mcq)} prompts):")
    print(f"  sweet-spot: {sum(1 <= r['num_correct'] < N_SAMPLES_PER_PROMPT for r in mcq)}")
    print(f"  uniform 0:  {sum(r['num_correct'] == 0 for r in mcq)}")
    print(f"  uniform N:  {sum(r['num_correct'] == N_SAMPLES_PER_PROMPT for r in mcq)}")
if ff:
    print(f"\nFree-form ({len(ff)} prompts):")
    print(f"  sweet-spot: {sum(1 <= r['num_correct'] < N_SAMPLES_PER_PROMPT for r in ff)}")
    print(f"  uniform 0:  {sum(r['num_correct'] == 0 for r in ff)}")
    print(f"  uniform N:  {sum(r['num_correct'] == N_SAMPLES_PER_PROMPT for r in ff)}")

# Clipping stats — high clipped rate among uniform-0 hints we should bump max tokens
clip_in_hard = [r for r in records if r["num_correct"] == 0 and any(c["clipped"] for c in r["completions"])]
if records:
    print(f"\nUniform-0 prompts with at least one clipped completion: {len(clip_in_hard)} "
          f"(of {sum(r['num_correct'] == 0 for r in records)} uniform-0)")

# Save filtered IDs for use by training notebook
sweet_ids = [r["id"] for r in sweet_spot]
with open(FILTERED_PATH, "w") as f:
    json.dump({
        "sweet_ids": sweet_ids,
        "n_samples_per_prompt": N_SAMPLES_PER_PROMPT,
        "distribution": dict(dist),
    }, f)
print(f"\nSaved {len(sweet_ids)} sweet-spot IDs to {FILTERED_PATH}")
print("Download this file and the difficulty_samples.jsonl from the Files panel.")